### Importe

In [1]:
import re
import json
import html
from pathlib import Path
from urllib.parse import quote

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


### Metadaten laden

Liest `merged_df.csv` (Ergebnis von `02_metadata_curate.ipynb`). Diese eine
Tabelle ersetzt die frueheren `df_01.csv` .. `df_10.csv` -- sie enthaelt
bereits alle Baende/Reihen inklusive der aus Anmerkungen/Antworten neu
angelegten Zeilen ("NEU-*"-IDs) sowie die Spalte `Textbeziehungen`
(intertextuelle Bezuege als "Typ: ID"-Paare, getrennt mit " | ").

In [3]:
MERGED_DF_PATH = Path("../../data/metadata/curated/merged_df.csv")
MANIFESTS_ROOT = Path("../../data/manifest/curated")

# Pfad-Praefix, unter dem die Manifeste vom Frontend aus (frontend/html/*.html)
# erreichbar sind -- relativ, da es kein "backend/static" gibt (nur
# "backend/data"); siehe frontend/js/register-personen.js, suche.js:
# PATH_* + book["manifest"]
VIEWER_MANIFEST_PREFIX = "../../backend/data/manifest/curated"

df = pd.read_csv(MERGED_DF_PATH, dtype=str)
df["pn"] = pd.to_numeric(df["Link"].str.extract(r"pn=(\d+)")[0], errors="coerce")

# Schneller Zugriff auf eine beliebige Zeile ueber ihre ID (fuer die Aufloesung
# der Textbeziehungen-Ziele, die in jedem beliebigen Band liegen koennen)
df_by_id = df.set_index("ID", drop=False)

print(f"{len(df)} Zeilen geladen, {df['Textbeziehungen'].notna().sum()} mit Textbeziehungen")


8434 Zeilen geladen, 984 mit Textbeziehungen


### Hilfsfunktionen: Manifest-/Canvas-IDs und Viewer-Links

Die Manifest- und Canvas-IDs folgen im gesamten Bestand demselben Schema
(`silo10!Bibliothek.tiff!{Schriftenreihe}!{Band}!tif`, Canvas `.../canvas/p{N}`
mit N = Bildnummer aus `Link` bzw. `pn=`). Das wurde vorab gegen alle 245
Manifeste geprueft (keine Abweichung) und erlaubt es, den Link zu einer
Zielzeile zu bauen, ohne deren Manifest-Datei oeffnen zu muessen.

Der Viewer-Link folgt demselben Muster wie an anderer Stelle im Frontend
(`register-personen.js`, `suche.js`): `viewer.html?manifest=...&canvas=...`.

In [5]:
def manifest_base_id(schriftenreihe, band):
    return (
        f"https://digilib.bbaw.de/digilib/Manifester/IIIF/3/"
        f"silo10!Bibliothek.tiff!{schriftenreihe}!{band}!tif"
    )


def canvas_uri(schriftenreihe, band, pn):
    return f"{manifest_base_id(schriftenreihe, band)}/canvas/p{int(pn)}"


def viewer_link(schriftenreihe, band, pn, text, escape=True):
    manifest_pfad = f"{VIEWER_MANIFEST_PREFIX}/{schriftenreihe}/{schriftenreihe}_{band}.json"
    href = (
        f"viewer.html?manifest={manifest_pfad}"
        f"&canvas={quote(canvas_uri(schriftenreihe, band, pn), safe='')}"
    )
    inhalt = html.escape(text) if escape else text
    return f'<a href="{href}" target="_blank" rel="noopener">{inhalt}</a>'


### Textbeziehungen zu klickbaren Links auflösen

`Textbeziehungen` enthaelt z.B. `"Fortsetzung: 094917590 | Vgl: 094928..."`.
Jede Ziel-ID wird ueber `df_by_id` aufgeloest: existiert eine gueltige
Seitenzahl (`pn`), wird ein klickbarer Link mit Titel gebaut (IIIF erlaubt
ein eingeschraenktes HTML-Subset -- inkl. `<a>` -- in Label-/Metadata-Werten;
Mirador rendert das, siehe `mirador-config.js` -> `theme.components.IIIFHtmlContent`).
Ohne aufloesbare Seitenzahl (z.B. bei den 18 NEU-Zeilen ohne Link) bleibt es
reiner Text, damit die Information nicht verloren geht.

In [7]:
def beziehungen_html(wert):
    if pd.isna(wert):
        return None

    # Typ-Span steckt INNERHALB des Links (bzw. direkt im Text ohne Link),
    # nicht daneben -- damit z.B. der Hover-Effekt des Links (Farbe etc.)
    # auch den Typ mit erfasst und beides als eine zusammengehoerige Einheit
    # wirkt.
    teile = []
    for eintrag in wert.split(" | "):
        typ, ziel_id = eintrag.split(": ", 1)
        typ, ziel_id = typ.strip(), ziel_id.strip()
        typ_span = f' <span class="beziehung-typ">({html.escape(typ)})</span>'

        ziel = df_by_id.loc[ziel_id] if ziel_id in df_by_id.index else None
        if ziel is not None and pd.notna(ziel["pn"]):
            inhalt = html.escape(ziel["Titel"]) + typ_span
            wert_html = viewer_link(ziel["Schriftenreihe"], ziel["Band"], ziel["pn"], inhalt, escape=False)
        else:
            titel = ziel["Titel"] if ziel is not None else ziel_id
            wert_html = html.escape(str(titel)) + typ_span

        teile.append(f'<span class="beziehung-wert">{wert_html}</span>')

    return "<br>".join(teile)


### Range-Funktion

Entspricht in der Grundlogik der bisherigen `process_all_manifests_in_folder`
(Bildstartseite aus `Link`/`pn=`, Seitenumfang aus Startseite/Endseite,
Fallback auf die naechste Zeile, wenn keine gueltige Seitenzahl vorliegt) --
neu ist, dass sie direkt auf `merged_df` statt auf den alten `df_01..df_10`
arbeitet und zusaetzlich `Textbeziehungen` (und `Abbilder`) als Metadata in
jede Range schreibt.

In [9]:
def extract_pn(link):
    if pd.isna(link):
        return None
    m = re.search(r"pn=(\d+)", str(link))
    return int(m.group(1)) if m else None


def baue_metadata(row):
    metadata = []

    if pd.notna(row.get("Autor")):
        metadata.append({"label": {"de": ["Autor"]}, "value": {"de": [str(row["Autor"]).strip()]}})

    # Jahr kommt aus dem RIS-Feld PY (per Werk, nicht aus dem Band abgeleitet --
    # praeziser, und bei buchweise statt jahrweise gezaehlten Reihen wie
    # 01-misc/04-phys die einzige Quelle fuer ein Jahr ueberhaupt)
    jahr = row.get("Jahr")
    if pd.notna(jahr):
        try:
            jahr_str = str(int(float(jahr)))
        except (ValueError, TypeError):
            jahr_str = None
        if jahr_str:
            metadata.append({"label": {"de": ["Erscheinungsjahr"]}, "value": {"de": [jahr_str]}})

    if pd.notna(row.get("Abbilder")):
        metadata.append({"label": {"de": ["Abbildungen"]}, "value": {"de": [str(row["Abbilder"]).strip()]}})

    beziehungen = beziehungen_html(row.get("Textbeziehungen"))
    if beziehungen:
        metadata.append({"label": {"de": ["Textbeziehungen"]}, "value": {"de": [beziehungen]}})

    return metadata


def enrich_manifest(manifest_path, df_band):
    with manifest_path.open("r", encoding="utf-8") as f:
        manifest = json.load(f)

    canvases = manifest.get("items", [])
    if not canvases:
        return 0

    df_band = df_band.copy()
    df_band["bild_start"] = df_band["Link"].apply(extract_pn)
    df_band = df_band.dropna(subset=["bild_start"]).sort_values("bild_start").reset_index(drop=True)
    if df_band.empty:
        return 0

    manifest_base = manifest.get("id", "https://example.org/manifest")
    structures = []
    anzahl_zeilen = len(df_band)

    for idx, row in df_band.iterrows():
        start_pn = int(row["bild_start"])

        try:
            sp = int(float(row["Startseite"]))
            ep = int(float(row["Endseite"]))
            end_pn = start_pn + (ep - sp)
            if end_pn > len(canvases):
                end_pn = len(canvases)
        except (ValueError, TypeError):
            if idx + 1 < anzahl_zeilen:
                end_pn = int(df_band.loc[idx + 1, "bild_start"]) - 1
            else:
                end_pn = len(canvases)

        start_index = start_pn - 1
        end_index = max(end_pn, start_pn)
        range_canvases = [{"id": c["id"], "type": "Canvas"} for c in canvases[start_index:end_index]]
        if not range_canvases:
            continue

        range_object = {
            "id": f"{manifest_base}/range/r_{row['ID']}",
            "type": "Range",
            "label": {"de": [str(row["Titel"])]},
            "items": range_canvases,
        }

        metadata = baue_metadata(row)
        if metadata:
            range_object["metadata"] = metadata

        structures.append(range_object)

    if not structures:
        return 0

    manifest["structures"] = structures
    with manifest_path.open("w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

    return len(structures)


### Alle Manifeste anreichern

Iteriert ueber alle Reihen-Ordner unter `data/manifest/curated/` (ersetzt die
zehn fast identischen Zellen der alten Version -- eine Schleife statt zehn
Kopien, damit neue Reihen/Baende automatisch mitlaufen).

In [11]:
gesamt_ranges = 0
gesamt_manifeste = 0

for sr_ordner in sorted(p for p in MANIFESTS_ROOT.iterdir() if p.is_dir()):
    print(f"=== {sr_ordner.name} ===")

    for manifest_path in sorted(sr_ordner.glob("*.json")):
        if "_mit_ranges" in manifest_path.name or manifest_path.name == "collection.json":
            continue

        band = manifest_path.stem.split("_", 1)[1]
        df_band = df[(df["Schriftenreihe"] == sr_ordner.name) & (df["Band"] == band)]

        if df_band.empty:
            print(f"  [!] Kein Eintrag fuer Band '{band}' ({manifest_path.name})")
            continue

        anzahl = enrich_manifest(manifest_path, df_band)
        gesamt_manifeste += 1
        if anzahl:
            gesamt_ranges += anzahl
            print(f"  [+] {manifest_path.name}: {anzahl} Ranges")
        else:
            print(f"  [-] {manifest_path.name}: keine gueltigen Ranges")

print(f"\nFertig: {gesamt_ranges} Ranges in {gesamt_manifeste} Manifesten.")


=== 01-misc ===
  [+] 01-misc_1.json: 57 Ranges
  [+] 01-misc_2.json: 26 Ranges
  [+] 01-misc_3.json: 58 Ranges
  [+] 01-misc_4.json: 54 Ranges
  [+] 01-misc_5.json: 31 Ranges
  [+] 01-misc_6.json: 42 Ranges
  [+] 01-misc_7-s.json: 1 Ranges
  [+] 01-misc_7.json: 19 Ranges
=== 02-hist ===
  [+] 02-hist_1745.json: 45 Ranges
  [+] 02-hist_1746.json: 29 Ranges
  [+] 02-hist_1747.json: 28 Ranges
  [+] 02-hist_1748.json: 29 Ranges
  [+] 02-hist_1749.json: 30 Ranges
  [+] 02-hist_1750.json: 25 Ranges
  [+] 02-hist_1751.json: 21 Ranges
  [+] 02-hist_1752.json: 22 Ranges
  [+] 02-hist_1753.json: 21 Ranges
  [+] 02-hist_1754.json: 27 Ranges
  [+] 02-hist_1755.json: 24 Ranges
  [+] 02-hist_1756.json: 23 Ranges
  [+] 02-hist_1757.json: 20 Ranges
  [+] 02-hist_1758.json: 23 Ranges
  [+] 02-hist_1759.json: 19 Ranges
  [+] 02-hist_1760.json: 22 Ranges
  [+] 02-hist_1761.json: 23 Ranges
  [+] 02-hist_1762.json: 25 Ranges
  [+] 02-hist_1763.json: 19 Ranges
  [+] 02-hist_1764.json: 24 Ranges
  [+] 02-hi